# Lab 2 · A composite value-momentum-quality signal

**Week 2 · Building a research workflow** · ≈ 90 minutes

Your universe is the 25 size × book-to-market portfolios for Asia-Pacific ex Japan, treated as 25
tradable "assets". You will build three signals point-in-time, clean and size-neutralise them, measure
their information coefficients (IC), and combine them without look-ahead.

**Data.** Monthly returns from the Kenneth R. French Data Library (Tuck School of Business, Dartmouth), *Asia Pacific ex Japan* region: Australia, **Hong Kong**, New Zealand and Singapore, July 1990 onwards. The setup cell downloads it once and caches it in `finex_data/`.

**How this lab works.** Run the setup cell, then work top to bottom. Each exercise has a `# TODO` cell followed by an automatic check (✓ / ✗ with a hint). When every check passes, the last cell prints a **completion code**: paste it into the course page to complete the lab.

In [ ]:
# FinEx Learning · lab setup. Run this cell first; it only defines helpers.
# Data: Kenneth R. French Data Library (Tuck School of Business, Dartmouth),
# Asia Pacific ex Japan series (Australia, Hong Kong, New Zealand, Singapore),
# downloaded on first use and cached next to the notebook.
import hashlib
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

FRENCH = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
CACHE = Path("finex_data")
CACHE.mkdir(exist_ok=True)


def _lines(name):
    path = CACHE / f"{name}.csv"
    if not path.exists():
        with urllib.request.urlopen(f"{FRENCH}{name}_CSV.zip", timeout=60) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
            path.write_bytes(archive.read(archive.namelist()[0]))
    return path.read_text(encoding="latin-1").splitlines()


def load_french(name, table=None, percent=True):
    """One monthly table from a French data library file, indexed by month.

    `table` is the title above the table (e.g. "Average Value Weighted Returns -- Monthly");
    None means the first monthly table. Returns are converted to decimals.
    """
    tables, title, current = {}, "", None
    for line in _lines(name):
        cells = [c.strip() for c in line.split(",")]
        if line.lstrip().startswith(","):
            current = []
            tables.setdefault(title or "first", (cells[1:], current))
            continue
        if current is not None and len(cells[0]) == 6 and cells[0].isdigit():
            current.append(cells)
            continue
        if line.strip():
            title, current = line.strip(), None
    key = table or next(k for k, (_, rows) in tables.items() if rows)
    header, rows = tables[key]
    frame = pd.DataFrame(
        [r[1:] for r in rows],
        columns=header,
        index=pd.PeriodIndex([r[0] for r in rows], freq="M"),
    ).astype(float).replace(-99.99, np.nan)
    return frame / 100 if percent else frame


def check(name, condition, hint):
    """Automatic check: passes quietly, or explains what to look at."""
    if not bool(condition):
        raise AssertionError(f"✗ {name}. Hint: {hint}")
    print(f"✓ {name}")


def completion_code(lab):
    code = hashlib.sha256(f"finex-quant-{lab}".encode()).hexdigest()[:6].upper()
    print(f"\nAll checks passed. Your completion code for the course page: FINEX-{lab.upper()}-{code}")


def annualise(returns):
    """Mean, volatility, Sharpe and t-stat of monthly excess returns, annualised."""
    r = returns.dropna()
    mean, vol = r.mean() * 12, r.std() * np.sqrt(12)
    return pd.Series({"mean": mean, "vol": vol, "sharpe": mean / vol,
                      "t_stat": r.mean() / r.std() * np.sqrt(len(r)), "months": len(r)})


print("Setup complete. Data source: Kenneth R. French Data Library · Asia Pacific ex Japan.")


In [ ]:
import warnings
warnings.filterwarnings("ignore")
from scipy.stats import spearmanr

returns = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Average Value Weighted Returns -- Monthly")
names = returns.columns
size_bucket = pd.Series([i // 5 + 1 for i in range(25)], index=names)   # 1 = small … 5 = big
value_bucket = pd.Series([i % 5 + 1 for i in range(25)], index=names)  # 1 = growth … 5 = value
returns.iloc[-3:, :6]

## Exercise 1 · A momentum signal with no look-ahead

At the end of month *t* the signal may only use returns up to *t*. Momentum is the compounded return
from month *t−11* to *t−1* (skip the latest month).

In [ ]:
def momentum_signal(r):
    # TODO: DataFrame of the same shape as r
    raise NotImplementedError("Exercise 1: write momentum_signal, then re-run this cell")

In [ ]:
full = momentum_signal(returns)
truncated = momentum_signal(returns.iloc[:200])
check("Momentum uses months t-11 … t-1",
      np.isclose(full.iloc[150, 0], (1 + returns.iloc[139:150, 0]).prod() - 1),
      "compound 11 months and shift by one so the latest month is skipped")
check("No look-ahead: future data does not change past signals",
      np.allclose(full.iloc[:200].dropna(), truncated.dropna()),
      "the signal at t must not use any return after t")

## Exercise 2 · Clean and neutralise

Value is the portfolio's book-to-market bucket, and *low volatility* (a defensive quality proxy) is minus
the trailing 36-month volatility. For each signal: z-score across the 25 assets each month, then remove
size by regressing on size-bucket dummies and keeping the residual, then z-score again.

In [ ]:
def zscore(df):
    return df.sub(df.mean(axis=1), axis=0).div(df.std(axis=1), axis=0)

dummies = pd.get_dummies(size_bucket).astype(float).values

In [ ]:
def size_neutral(df):
    # TODO: per month, regress the row on `dummies` and keep the residual
    raise NotImplementedError("Exercise 2: write size_neutral, then re-run this cell")

value_raw = pd.DataFrame(np.tile(value_bucket.values, (len(returns), 1)), index=returns.index, columns=names).astype(float)
signals = {
    "momentum": None,  # TODO
    "value": None,     # TODO
    "low_vol": None,   # TODO
}

In [ ]:
last = signals["momentum"].dropna().iloc[-1]
by_size = last.groupby(size_bucket).mean()
check("Signals are size-neutral", np.allclose(by_size, 0, atol=1e-8),
      "after neutralising, the average signal within every size bucket should be zero")
check("All three signals are present",
      all(isinstance(signals[k], pd.DataFrame) for k in ["momentum", "value", "low_vol"]),
      "fill in every entry of `signals`")

## Exercise 3 · Rank information coefficient

The rank IC at month *t* is the Spearman correlation between the signal at *t* and the return in *t+1*.
Report the mean IC and its t-statistic for each signal.

In [ ]:
def rank_ic(signal, horizon=1):
    # TODO: Series of monthly Spearman ICs against the forward `horizon`-month return
    raise NotImplementedError("Exercise 3: write rank_ic, then re-run this cell")

ic_table = None  # TODO: DataFrame with rows = signals, columns = mean_ic, t_stat

In [ ]:
check("IC table covers the three signals",
      isinstance(ic_table, pd.DataFrame) and set(ic_table.index) == {"momentum", "value", "low_vol"},
      "one row per signal")
check("Every signal has a positive mean IC", (ic_table["mean_ic"] > 0).all(),
      "check the sign: higher signal should mean higher next-month return")

## Exercise 4 · Combine without peeking

Weight each signal by its **average IC up to the previous month** (an expanding mean, shifted by one),
so the weights never use information from the future. Measure the composite's IC.

In [ ]:
composite = None      # TODO: DataFrame, IC-weighted sum of the three signals
composite_ic = None   # TODO: Series of monthly rank ICs of the composite

In [ ]:
check("Composite IC is positive and significant",
      composite_ic is not None and composite_ic.mean() > 0 and composite_ic.mean() / composite_ic.std() * np.sqrt(len(composite_ic)) > 3,
      "weights from an expanding mean of past ICs, shifted by one month")
completion_code("lab2")

## Reflect

* Value's IC *rises* with the horizon while momentum's stays flat. What does that say about how often each should be rebalanced?
* Why does the composite need `shift(1)` on its weights? What would happen to its IC without it?
* These are portfolios, not stocks. What would change with a real stock universe of Hong Kong names?